In [ ]:
%pip install openmeteo_requests

In [25]:
import openmeteo_requests
from datetime import datetime

class IncreaseSpeed:
    def __init__(self, current_speed: int, max_speed: int, upper_border: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.upper_border = min(upper_border, max_speed)
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.upper_border:
            raise StopIteration
        
        new_speed = self.current_speed + self.step
        if new_speed > self.upper_border:
            new_speed = self.upper_border
        
        self.current_speed = new_speed
        return self.current_speed

class DecreaseSpeed:
    def __init__(self, current_speed: int, lower_border: int, step=10):
        self.current_speed = current_speed
        self.lower_border = max(0, lower_border)
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.lower_border:
            raise StopIteration
        
        new_speed = self.current_speed - self.step
        if new_speed < self.lower_border:
            new_speed = self.lower_border
            
        self.current_speed = new_speed
        return self.current_speed

class Car:
    _total_cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max_speed
        self.current_speed = current_speed
        self.state = self.current_speed > 0
        
        if self.state:
            Car._total_cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        if upper_border is None:
            upper_border = self.current_speed + step
            
        old_speed = self.current_speed
        increaser = IncreaseSpeed(self.current_speed, self.max_speed, upper_border, step)
        
        if not self.state and upper_border > 0:
            self.state = True
            Car._total_cars_on_road += 1

        for speed in increaser:
            if speed < upper_border:
                print(">> INFO: Speed increases by 10")
            self.current_speed = speed

        print(f">> INFO: The speed of this car has been increased from {old_speed} to {self.current_speed}")

    def brake(self, lower_border=None, step=10):
        if lower_border is None:
            lower_border = max(0, self.current_speed - step)
            
        old_speed = self.current_speed
        decreaser = DecreaseSpeed(self.current_speed, lower_border, step)

        for speed in decreaser:
            if speed > lower_border:
                print(">> INFO: Speed decreases by 10")
            self.current_speed = speed

        print(f">> INFO: The speed of this car has been decreased from {old_speed} to {self.current_speed}")

    def parking(self):
        if not self.state:
            print(">> INFO: Car is already parked/off road.")
            return

        if self.current_speed > 0:
            self.brake(0)
            
        print(">> Parking the car...")
        self.state = False
        Car._total_cars_on_road -= 1

    @classmethod
    def total_cars(cls):
        return cls._total_cars_on_road

    @staticmethod
    def show_weather():
        try:
            openmeteo = openmeteo_requests.Client()
            url = "http://api.open-meteo.com/v1/forecast"
            params = {
                "latitude": 59.9386,
                "longitude": 30.3141,
                "current": ["temperature_2m", "apparent_temperature", "rain", "wind_speed_10m"],
                "wind_speed_unit": "ms",
                "timezone": "Europe/Moscow"
            }

            responses = openmeteo.weather_api(url, params=params)
            response = responses[0]

            current = response.Current()
            print(f">> Current temperature: {round(current.Variables(0).Value(), 1)} C")
            print(f">> Current apparent_temperature: {round(current.Variables(1).Value(), 1)} C")
            print(f">> Current rain: {current.Variables(2).Value()} mm")
            print(f">> Current wind_speed: {round(current.Variables(3).Value(), 1)} m/s")
        except Exception as e:
            print(f">> INFO: Could not fetch weather data. Error: {e}")

In [26]:
car1 = Car(100, 20)
car2 = Car(60, 30)
car3 = Car(100, 0)
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [27]:
car1.accelerate(100)
car2.accelerate(50)

print("Speed of car 1:", car1.current_speed)
print("Speed of car 2:", car2.current_speed)

>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: The speed of this car has been increased from 20 to 100
>> INFO: Speed increases by 10
>> INFO: The speed of this car has been increased from 30 to 50
Speed of car 1: 100
Speed of car 2: 50


In [28]:
car1.brake(10)

car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: The speed of this car has been decreased from 100 to 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: Speed decreases by 10
>> INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
>> Parking the car...
Total cars on road: 1


In [29]:
car3.accelerate(80)
car3.show_weather()
print("Total cars on road:", Car.total_cars())

>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: Speed increases by 10
>> INFO: The speed of this car has been increased from 0 to 80


>> Current temperature: 3.6 C
>> Current apparent_temperature: -0.3 C
>> Current rain: 0.0 mm
>> Current wind_speed: 3.8 m/s
Total cars on road: 2


In [30]:
car2.accelerate(10)
print("Total cars on road:", Car.total_cars())

>> INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [31]:
Car.show_weather()

>> Current temperature: 3.6 C
>> Current apparent_temperature: -0.3 C
>> Current rain: 0.0 mm
>> Current wind_speed: 3.8 m/s
